# Optimizer Agent Notebook

This notebook demonstrates the Optimizer Agent's capabilities in improving quantum circuits.

## Purpose
The Optimizer Agent refines generated circuits to improve performance and efficiency. This notebook covers:

1.  **Heuristic Optimization**: Applying standard Cirq optimizations like gate merging and dropping negligible operations.
2.  **RL Optimization**: Using a Reinforcement Learning loop to iteratively improve circuit metrics (depth, gate count) based on a reward function.
3.  **Comparisoon**: Analyzing and comparing the metrics of the original vs. optimized circuits.

## Usage
Run this notebook to optimize existing Cirq circuits and observe the improvements in circuit depth and gate count.


In [1]:
import sys
import os
from pathlib import Path
import cirq

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.cirq_rag_code_assistant.config import get_config, setup_logging
from src.agents.optimizer import OptimizerAgent

# Setup logging
setup_logging()

2026-03-26 11:22:28 | INFO     | src.cirq_rag_code_assistant.config.logging:setup_all:127 | Logging configuration completed


### Initialize Agent
Initialize the Optimizer Agent.

In [2]:
# First, add these imports and setup the RAG components
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever

# Initialize Knowledge Base with all datasets
KNOWLEDGE_BASE_DIR = project_root / "data" / "knowledge_base"
kb = KnowledgeBase(knowledge_base_path=KNOWLEDGE_BASE_DIR)
kb.load_from_directory()
kb.index_entries()

# Create retriever
retriever = Retriever(knowledge_base=kb)

# Initialize optimizer with RAG retriever
optimizer = OptimizerAgent(retriever=retriever)
print("Optimizer Agent initialized with RAG support.")

2026-03-26 11:22:28 | INFO     | config.config_loader:load:101 | ✅ Loaded configuration from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\config\config.json


2026-03-26 11:22:28,280 - botocore.credentials - INFO - credentials.py:1252 - Found credentials in environment variables.


2026-03-26 11:22:28 | INFO     | src.rag.embeddings:_init_aws:124 | Using AWS Bedrock Embeddings: amazon.nova-2-multimodal-embeddings-v1:0, dim=1024
2026-03-26 11:22:28 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2026-03-26 11:22:28 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2026-03-26 11:22:28 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 1024
2026-03-26 11:22:28 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2026-03-26 11:22:28 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading knowledge base from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\knowledge_base\curated_designer_examples.jsonl
2026-03-26 11:22:28 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 | Loaded 107 entries from D:\University\Uni\Huawei ICT Compeition\Cirq-RAG-Code-Assistant\data\knowledge_base\curated_designer_examples.js

Optimizer Agent initialized with RAG support.


### Create Sample Circuit
Let's create an unoptimized circuit to test.

In [3]:
# Create a circuit with redundant gates
q0, q1 = cirq.LineQubit.range(2)
circuit = cirq.Circuit(
    cirq.X(q0),
    cirq.X(q0),  # Redundant (Identity)
    cirq.Z(q1),
    cirq.Z(q1),  # Redundant (Identity)
    cirq.CNOT(q0, q1),
    cirq.CNOT(q0, q1), # Redundant (Identity)
    cirq.H(q0),
    cirq.H(q0)   # Redundant (Identity)
)

print("Original Circuit:")
print(circuit)

Original Circuit:
0: ───X───X───@───@───H───H───
              │   │
1: ───Z───Z───X───X───────────


### Optimize Circuit
Now let's run the optimizer.

In [4]:
task = {
    "circuit": circuit,
    "optimization_level": "aggressive",
    "use_rl": True  # Enable RL optimization
}

try:
    result = optimizer.execute(task)
    
    if result['success']:
        print("Successfully optimized circuit!")
        print("\nOptimized Circuit:")
        print("-" * 40)
        print(result['optimized_code'])
        print("-" * 40)
        
        print("\nMetrics Comparison:")
        print(f"Original Depth: {result['original_metrics'].get('depth')}")
        print(f"Optimized Depth: {result['optimized_metrics'].get('depth')}")
        print(f"Original Gate Count: {result['original_metrics'].get('num_operations')}")
        print(f"Optimized Gate Count: {result['optimized_metrics'].get('num_operations')}")
        
        print("\nImprovements:")
        for imp in result['improvements']:
            print(f"- {imp}")
            
    else:
        print(f"Failed to optimize circuit: {result.get('error')}")
        
except Exception as e:
    print(f"Error executing task: {e}")

2026-03-26 11:35:32 | INFO     | src.agents.optimizer:execute:248 | Retrieved 3 optimization references from RAG


Successfully optimized circuit!

Optimized Circuit:
----------------------------------------
import cirq

circuit = cirq.Circuit([
    cirq.Moment(
        cirq.X(cirq.LineQubit(0)),
        cirq.Z(cirq.LineQubit(1)),
    ),
    cirq.Moment(
        cirq.X(cirq.LineQubit(0)),
        cirq.Z(cirq.LineQubit(1)),
    ),
    cirq.Moment(
        cirq.CNOT(cirq.LineQubit(0), cirq.LineQubit(1)),
    ),
    cirq.Moment(
        cirq.CNOT(cirq.LineQubit(0), cirq.LineQubit(1)),
    ),
    cirq.Moment(
        cirq.H(cirq.LineQubit(0)),
    ),
    cirq.Moment(
        cirq.H(cirq.LineQubit(0)),
    ),
])

print(circuit)
----------------------------------------

Metrics Comparison:
Original Depth: 6
Optimized Depth: 6
Original Gate Count: 8
Optimized Gate Count: 8

Improvements:


### LLM Optimization
Use the LLM backend to optimize the circuit.

In [5]:
llm_task = {
    "circuit": circuit,
    "use_llm": True,
    "use_heuristics": False,
    "use_rl": False
}

try:
    print("Running LLM Optimization...")
    llm_result = optimizer.execute(llm_task)
    
    if llm_result['success']:
        print("Successfully optimized circuit using LLM!")
        print("\nLLM Optimized Circuit:")
        print("-" * 40)
        print(llm_result['optimized_code'])
        print("-" * 40)
        
        print("\nLLM Metrics:")
        print(f"Depth: {llm_result['optimized_metrics'].get('depth')}")
        print(f"Gate Count: {llm_result['optimized_metrics'].get('num_operations')}")
    else:
        print(f"LLM optimization failed: {llm_result.get('error')}")

except Exception as e:
    print(f"Error executing LLM task: {e}")

Running LLM Optimization...


2026-03-26 11:35:32 | INFO     | src.agents.optimizer:execute:248 | Retrieved 3 optimization references from RAG
2026-03-26 11:35:32 | INFO     | src.agents.optimizer:execute:254 | Running LLM optimization
2026-03-26 11:35:32 | INFO     | src.rag.generator:generate_direct:561 | Generating code directly (no RAG) using aws/arn:aws:bedrock:us-east-1:626230557862:application-inference-profile/9iiyojvjl773
2026-03-26 11:35:38 | INFO     | src.rag.generator:generate_direct:634 | ✅ Direct code generation completed


Successfully optimized circuit using LLM!

LLM Optimized Circuit:
----------------------------------------
import cirq

# All pairs of identical gates cancel out (X·X=I, Z·Z=I, CNOT·CNOT=I, H·H=I)
# The optimized circuit is empty (identity operation)
circuit = cirq.Circuit()

print(circuit)
----------------------------------------

LLM Metrics:
Depth: 0
Gate Count: 0
